# AI Resume Analyzer and Career Assistant using NLP and Generative AI
### **NVIDIA AI Internship Capstone Project Submission**

This notebook demonstrates the end-to-end pipeline of the **AI Resume Analyzer and Career Assistant**. It parses resume PDFs, extracts technical skills, performs semantic similarity matching against job descriptions, and uses the **Hugging Face FLAN-T5** model to generate personalized recommendations and interview questions.

### **Pipeline Steps:**
1. **Environment Setup**: Install dependencies & check GPU/CUDA status.
2. **Text Extraction**: Parse raw text from resume PDFs using `pdfplumber` / `pypdf`.
3. **Semantic Embedding**: Embed resume and job description using Sentence Transformers on GPU, calculating compatibility via Cosine Similarity.
4. **Skill Mapping**: Extract matched and missing skills utilizing regex taxonomy parsing.
5. **Generative Career Recommendations**: Run FLAN-T5 for improvement tips, rewrites, and interview preparation.
6. **Interactive Dashboard**: Run the Streamlit UI inside Google Colab using a sharing service.

## 1. Environment Setup
First, we will install all the necessary libraries and verify that we have access to GPU hardware acceleration.

In [ ]:
# Install dependencies
!pip install -q streamlit sentence-transformers transformers pypdf pdfplumber reportlab pandas numpy matplotlib

In [ ]:
import torch
import sys
import os

# Check PyTorch CUDA acceleration status
print(f"Python Version: {sys.version}")
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device: {torch.cuda.get_device_name(0)}")
    print(f"CUDA Device Capability: {torch.cuda.get_device_capability(0)}")
    print(f"Current CUDA Device Index: {torch.cuda.current_device()}")
else:
    print("WARNING: Running on CPU. Recommendations and model embeddings will have higher latency. Change Colab runtime settings to T4 GPU for acceleration.")

## 2. Directory and File Structure Creation
Let's write our project files directly to the Colab workspace. This replicates the structure of our modular codebase.

In [ ]:
# Create necessary directories
!mkdir -p src sample_data docs models

In [ ]:
%%writefile src/utils.py
import torch
import logging
import sys

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(name)s - %(levelname)s - %(message)s', handlers=[logging.StreamHandler(sys.stdout)])
logger = logging.getLogger("AI-Resume-Analyzer")

def get_gpu_status():
    status = {
        "available": False,
        "device_count": 0,
        "device_name": "CPU",
        "device_type": "cpu",
        "acceleration_status": "CPU (No CUDA acceleration available)",
        "vram_allocated_gb": 0.0,
        "vram_reserved_gb": 0.0
    }
    if torch.cuda.is_available():
        status["available"] = True
        status["device_count"] = torch.cuda.device_count()
        status["device_name"] = torch.cuda.get_device_name(0)
        status["device_type"] = "cuda"
        status["acceleration_status"] = "CUDA Enabled (NVIDIA GPU Accelerated)"
        try:
            status["vram_allocated_gb"] = round(torch.cuda.memory_allocated(0) / (1024 ** 3), 2)
            status["vram_reserved_gb"] = round(torch.cuda.memory_reserved(0) / (1024 ** 3), 2)
        except:
            pass
    return status

def log_info(msg): logger.info(msg)
def log_error(msg, exc=None): logger.error(f"{msg} - {str(exc)}" if exc else msg)

In [ ]:
%%writefile src/pdf_reader.py
import re
import pdfplumber
from pypdf import PdfReader
from src.utils import log_info, log_error

def clean_text(text):
    if not text: return ""
    text = re.sub(r'\s+', ' ', text)
    text = "".join(ch for ch in text if ch.isprintable() or ch in ['\n', '\t'])
    return text.strip()

def extract_text_from_pdf(pdf_path):
    text = ""
    try:
        with pdfplumber.open(pdf_path) as pdf:
            pages_text = [page.extract_text() for page in pdf.pages if page.extract_text()]
            text = "\n".join(pages_text)
        if text.strip(): return clean_text(text)
    except Exception as e:
        log_error("pdfplumber failed", e)
    try:
        reader = PdfReader(pdf_path)
        pages_text = [page.extract_text() for page in reader.pages if page.extract_text()]
        text = "\n".join(pages_text)
        if text.strip(): return clean_text(text)
    except Exception as e:
        log_error("pypdf failed", e)
    raise ValueError("Could not extract text from PDF.")

In [ ]:
%%writefile src/embeddings.py
import torch
from sentence_transformers import SentenceTransformer, util
from src.utils import log_info, get_gpu_status

class ResumeMatcher:
    def __init__(self, model_name='all-MiniLM-L6-v2'):
        gpu_status = get_gpu_status()
        self.device = "cuda" if gpu_status["available"] else "cpu"
        self.model = SentenceTransformer(model_name, device=self.device)
        log_info(f"Loaded embeddings model '{model_name}' on {self.device}")
        
    def compute_similarity(self, resume_text, job_desc_text):
        if not resume_text.strip() or not job_desc_text.strip(): return 0.0
        resume_emb = self.model.encode(resume_text, convert_to_tensor=True, show_progress_bar=False)
        job_emb = self.model.encode(job_desc_text, convert_to_tensor=True, show_progress_bar=False)
        cosine_score = util.cos_sim(resume_emb, job_emb)
        return round(max(0.0, min(100.0, float(cosine_score[0][0]) * 100)), 2)

In [ ]:
%%writefile src/skill_analyzer.py
import re
from src.utils import log_info

SKILL_TAXONOMY = {
    "Python": [r"\bpython\b"],
    "C++": [r"\bc\+\+\b", r"\bcpp\b"],
    "SQL": [r"\bsql\b", r"\bmysql\b", r"\bpostgresql\b"],
    "Machine Learning": [r"\bmachine\s+learning\b", r"\bml\b"],
    "Deep Learning": [r"\bdeep\s+learning\b", r"\bdl\b"],
    "Natural Language Processing": [r"\bnatural\s+language\s+processing\b", r"\bnlp\b"],
    "Computer Vision": [r"\bcomputer\s+vision\b", r"\bcv\b"],
    "Transformers": [r"\btransformers?\b"],
    "PyTorch": [r"\bpy\s*torch\b"],
    "TensorFlow": [r"\btensor\s*flow\b"],
    "Scikit-Learn": [r"\bscikit-learn\b", r"\bsklearn\b"],
    "Pandas": [r"\bpandas\b"],
    "Hugging Face": [r"\bhugging\s*face\b", r"\bhf\b"],
    "CUDA": [r"\bcuda\b"],
    "TensorRT": [r"\btensorrt\b"],
    "Docker": [r"\bdocker\b"],
    "Kubernetes": [r"\bkubernetes\b", r"\bk8s\b"],
    "Git": [r"\bgit\b", r"\bgithub\b"],
    "CI/CD": [r"\bci/cd\b"],
    "Linux": [r"\blinux\b", r"\bubuntu\b"],
    "Agile": [r"\bagile\b", r"\bscrum\b"]
}

def extract_skills_from_text(text):
    found_skills = set()
    if not text: return found_skills
    text_lower = text.lower()
    for skill, patterns in SKILL_TAXONOMY.items():
        for pattern in patterns:
            if re.search(pattern, text_lower):
                found_skills.add(skill)
                break
    return found_skills

def analyze_skills(resume_text, job_desc_text):
    resume_skills = extract_skills_from_text(resume_text)
    job_skills = extract_skills_from_text(job_desc_text)
    matching_skills = resume_skills.intersection(job_skills)
    missing_skills = job_skills.difference(resume_skills)
    candidate_skills = resume_skills.difference(job_skills)
    skills_match_score = round((len(matching_skills) / len(job_skills) * 100), 2) if job_skills else 100.0
    return {
        "resume_skills": sorted(list(resume_skills)),
        "job_skills": sorted(list(job_skills)),
        "matching_skills": sorted(list(matching_skills)),
        "missing_skills": sorted(list(missing_skills)),
        "candidate_skills": sorted(list(candidate_skills)),
        "skills_match_score": skills_match_score
    }

In [ ]:
%%writefile src/recommender.py
import torch
from transformers import pipeline, AutoModelForSeq2SeqLM, AutoTokenizer
from src.utils import log_info, log_error, get_gpu_status

class CareerRecommender:
    def __init__(self, model_name='google/flan-t5-base', use_llm=True):
        self.use_llm = use_llm
        self.pipeline = None
        if self.use_llm:
            try:
                gpu = get_gpu_status()["available"]
                device = 0 if gpu else -1
                tokenizer = AutoTokenizer.from_pretrained(model_name)
                model = AutoModelForSeq2SeqLM.from_pretrained(
                    model_name, 
                    torch_dtype=torch.float16 if gpu else torch.float32,
                    device_map="auto" if gpu else None
                )
                self.pipeline = pipeline("text2text-generation", model=model, tokenizer=tokenizer, device=device if not gpu else None)
                log_info(f"Loaded LLM pipeline on GPU={gpu}")
            except Exception as e:
                log_error("LLM load failed, using rules", e)
                self.use_llm = False
                
    def generate_improvement_tips(self, missing_skills, matching_skills):
        if not missing_skills: return ["No missing skills detected. Excellent!"]
        if self.use_llm and self.pipeline:
            prompt = f"Skills missing: {', '.join(missing_skills)}. Matching: {', '.join(matching_skills)}. Write a list of 3 resume tips."
            try:
                out = self.pipeline(prompt, max_length=150, temperature=0.7, do_sample=True)[0]['generated_text']
                return [t.strip().lstrip('-*•').strip() for t in out.split('\n') if t.strip()]
            except: pass
        return [f"Add a hands-on project utilizing {s} to bridge your expertise gap." for s in missing_skills[:3]]
        
    def generate_wording_suggestions(self, missing_skills):
        if not missing_skills: return ["Profile wording aligns with industry standards."]
        if self.use_llm and self.pipeline:
            prompt = f"Give 2 wording improvements to include these skills in a resume: {', '.join(missing_skills[:2])}. format: 'Before: -> After:'"
            try:
                out = self.pipeline(prompt, max_length=150, temperature=0.7, do_sample=True)[0]['generated_text']
                return [w.strip() for w in out.split('\n') if w.strip()]
            except: pass
        wording = {
            "CUDA": "Before: 'Did programming on GPUs.' -> After: 'Wrote efficient, parallelized kernels using CUDA, decreasing training times by 40%.'",
            "Docker": "Before: 'Set up Docker.' -> After: 'Containerized model microservices using Docker for streamlined multi-environment deployment.'"
        }
        return [wording.get(s, f"Rewrite bullets to explicitly mention using {s} in project implementations.") for s in missing_skills[:2]]

    def generate_interview_questions(self, matching_skills, missing_skills):
        if self.use_llm and self.pipeline:
            prompt = f"Write 2 technical questions on: {', '.join(matching_skills[:1])} and 1 on missing skill {', '.join(missing_skills[:1])}. Add 1 HR behavioral question."
            try:
                out = self.pipeline(prompt, max_length=200, temperature=0.7, do_sample=True)[0]['generated_text']
                qs = [q.strip() for q in out.split('\n') if q.strip()]
                return {"technical": qs[:2], "hr": [qs[2]] if len(qs)>2 else ["Explain how you handle tight project deadlines."], "project": ["Explain your project architecture."]}
            except: pass
        return {
            "technical": [f"Explain a key design optimization you did in {s}." for s in (matching_skills[:2] or ["Machine Learning"])],
            "hr": ["Describe a time you resolved a major disagreement in a technical team."],
            "project": ["Walk us through the data ETL and deployment pipeline of a project you built."]
        }

## 3. Generate Sample Documents
We will run a script to generate a high-quality sample resume PDF and target job description txt.

In [ ]:
from reportlab.lib.pagesizes import letter
from reportlab.pdfgen import canvas

def create_samples():
    # 1. Create PDF Resume
    c = canvas.Canvas("sample_data/sample_resume.pdf", pagesize=letter)
    c.setFont("Helvetica-Bold", 20)
    c.drawString(50, 750, "John Doe - ML Developer")
    c.setFont("Helvetica", 10)
    c.drawString(50, 735, "Email: john@doe.com | Skills: Python, Machine Learning, TensorFlow, Git, SQL, Agile")
    c.drawString(50, 700, "Experience: Built churn models with Scikit-Learn and Pandas. Utilized Git and SQL daily.")
    c.drawString(50, 670, "Projects: Created text search app using Hugging Face transformers. Deployed via Flask.")
    c.save()
    
    # 2. Create Job Description
    jd = """Position: NVIDIA Machine Learning Engineer
Requirements:
- Python, PyTorch, Deep Learning, Transformers, Git, SQL
- CUDA programming, GPU Acceleration
- Docker containerization, Kubernetes orchestrations"""
    with open("sample_data/sample_job_description.txt", "w") as f:
        f.write(jd)
        
    print("Sample files generated at sample_data/")

create_samples()

## 4. Run Core Pipeline Demonstration
Let's test loading our models, parsing the PDF, running similarity matching, analyzing skills, and generating recommendations on the GPU.

In [ ]:
from src.pdf_reader import extract_text_from_pdf
from src.embeddings import ResumeMatcher
from src.skill_analyzer import analyze_skills
from src.recommender import CareerRecommender
import time

# 1. Parse PDF
resume_text = extract_text_from_pdf("sample_data/sample_resume.pdf")
with open("sample_data/sample_job_description.txt", "r") as f:
    job_desc = f.read()
    
# 2. Compute similarity & measure execution times
matcher = ResumeMatcher()

start_time = time.time()
similarity = matcher.compute_similarity(resume_text, job_desc)
end_time = time.time()
print(f"\nResume Match Score: {similarity}%")
print(f"Inference latency: {end_time - start_time:.4f} seconds")

# 3. Run Skill Extraction
skills_results = analyze_skills(resume_text, job_desc)
print(f"\nMatching Skills: {skills_results['matching_skills']}")
print(f"Missing Skills: {skills_results['missing_skills']}")
print(f"Skills Match Coverage Score: {skills_results['skills_match_score']}%")

# 4. Generate Recommendations (T5 LLM)
recommender = CareerRecommender(use_llm=True)

start_gen = time.time()
tips = recommender.generate_improvement_tips(skills_results['missing_skills'], skills_results['matching_skills'])
wording = recommender.generate_wording_suggestions(skills_results['missing_skills'])
questions = recommender.generate_interview_questions(skills_results['matching_skills'], skills_results['missing_skills'])
end_gen = time.time()

print(f"\nGenerative inference latency: {end_gen - start_gen:.4f} seconds")
print("\n--- RESUME IMPROVEMENT TIPS ---")
for t in tips: print(f"- {t}")
print("\n--- WORDING UPGRADES ---")
for w in wording: print(f"- {w}")
print("\n--- INTERVIEW QUESTIONS ---")
for category, qs in questions.items():
    print(f"{category.upper()}:")
    for q in qs: print(f"  * {q}")

## 5. Performance Latency Comparison: CPU vs. GPU
Let's compare execution latency on CPU vs GPU for the model embeddings and text generation.

In [ ]:
import time
from sentence_transformers import SentenceTransformer

print("Comparing Sentence Transformer latencies:")

# CPU Latency
cpu_model = SentenceTransformer('all-MiniLM-L6-v2', device='cpu')
t0 = time.time()
for _ in range(5):
    cpu_model.encode(resume_text)
cpu_time = (time.time() - t0) / 5
print(f"Average CPU encoding latency: {cpu_time:.4f} seconds")

# GPU Latency
if torch.cuda.is_available():
    gpu_model = SentenceTransformer('all-MiniLM-L6-v2', device='cuda')
    t0 = time.time()
    for _ in range(5):
        gpu_model.encode(resume_text)
    gpu_time = (time.time() - t0) / 5
    print(f"Average GPU encoding latency: {gpu_time:.4f} seconds")
    print(f"GPU speedup factor: {cpu_time / gpu_time:.1f}x")
else:
    print("GPU not available for speedup comparison.")

## 6. Launch the Streamlit App from Google Colab
To run the interactive Streamlit dashboard directly from your Colab environment:
1. Run the cell below to write the Streamlit runner code `app.py`.
2. Start the Streamlit server and expose it to the internet using a public tunnel (like `localtunnel`).
3. Click the provided URL link, and input the Colab IP (displayed below) if prompted for security access.

In [ ]:
# Fetch public IP for tunnel password verification
!curl -s ipv4.icanhazip.com

In [ ]:
# Run Streamlit in the background and expose via localtunnel
!streamlit run app.py & npx localtunnel --port 8501